# Семинар 5. Агрегация в SQL

На этом занятии вы будете считать сводные показатели по демобазе авиаперевозок:
количество, сумму, среднее, минимум и максимум — по всей таблице и по группам строк,
отбирать группы и сравнивать строку с агрегатом.

**Как устроен ноутбук.** Каждая задача — описание и ячейка, которая начинается с `%%sql`.
Всё, что написано в такой ячейке ниже первой строки, отправляется в PostgreSQL,
а результат показывается таблицей.

**База.** PostgreSQL 17 в Docker, демобаза авиаперевозок Postgres Pro (версия 2016 года).
Как поднять базу, открыть ноутбук и подключиться через DBeaver, написано в файле `README.md`
в корне репозитория.

Все таблицы лежат в схеме `bookings`, в запросах пишем имя вместе со схемой: `bookings.flights`.

### Таблица демобазы, которая понадобится

Все задачи — по таблице `bookings.flights`: одна строка — рейс в конкретный день.

| Столбец | Что в нём |
|---|---|
| `flight_id` | идентификатор рейса |
| `flight_no` | номер рейса, один и тот же каждый день |
| `departure_airport`, `arrival_airport` | коды аэропортов вылета и прилёта |
| `status` | статус: `Scheduled`, `On Time`, `Delayed`, `Departed`, `Arrived`, `Cancelled` |
| `scheduled_departure` | вылет по расписанию |
| `actual_departure` | фактический вылет; пусто (`NULL`), если рейс ещё не вылетел |

Данные сформированы на момент 13.10.2016 17:00: у более поздних рейсов фактического вылета ещё нет.

**Вылетевший рейс** — рейс, у которого заполнен `actual_departure`, каким бы ни был его статус.

**Схема демобазы.** Таблицы, их ключи и связи; подробное описание столбцов — в [`db/schema.md`](../db/schema.md).

![Схема демобазы авиаперевозок](../db/schema.svg)

In [ ]:
import sys; sys.path.insert(0, "../jupyter")   # магия %%sql лежит в папке jupyter репозитория
%load_ext sql_magic

### Проверка подключения
Если ячейка ниже показала пять рейсов — всё готово к работе.

In [ ]:
%%sql
SELECT * FROM bookings.flights LIMIT 5;

# Задачи: задержки вылетов

Служба пунктуальности авиакомпании изучает задержки вылетов: как они меняются со временем,
в каких аэропортах они больше и какие рейсы задержались сильнее всего.

**Задержка вылета** — разница между фактическим вылетом и вылетом по расписанию.
Во всех задачах она считается **в минутах** и по всем вылетевшим рейсам — тем, у которых
заполнен `actual_departure`, независимо от статуса.

**Памятка**

```sql
SELECT столбец, АГРЕГАТ(выражение) AS имя   -- что вывести
FROM bookings.таблица                       -- откуда
WHERE условие                               -- какие строки взять до группировки
GROUP BY столбец                            -- по чему группировать
HAVING условие на агрегат                   -- какие группы оставить
ORDER BY столбец DESC                       -- как отсортировать
LIMIT 10;                                   -- сколько строк вывести
```

| Что | Как пишется |
|---|---|
| Задержка вылета в минутах | `ROUND(EXTRACT(EPOCH FROM actual_departure - scheduled_departure) / 60)` |
| Начало недели (понедельник) | `date_trunc('week', scheduled_departure)::date` |
| Округление | `ROUND(x, 1)` — до одного знака, `ROUND(x)::int` — до целого |
| Агрегат по части строк | `COUNT(*) FILTER (WHERE условие)` или `SUM(CASE WHEN условие THEN 1 ELSE 0 END)` |
| Агрегат рядом со строкой | `(SELECT АГРЕГАТ(…) FROM … WHERE …) AS имя` — подзапрос в `SELECT` |

`EXTRACT(EPOCH FROM интервал)` — длина интервала в секундах; делим на 60 и получаем минуты,
`ROUND` убирает дробную часть.

### Задача 1. Общая картина

Прежде чем считать задержки, посмотрим, какие данные есть. Одним запросом по таблице рейсов найдите:

- **cnt_flights** — сколько всего рейсов;
- **cnt_departed** — сколько из них уже вылетело (фактический вылет заполнен);
- **cnt_airports** — из скольких разных аэропортов есть вылеты;
- **first_departure** и **last_departure** — самый ранний и самый поздний вылет по расписанию.

In [ ]:
%%sql
-- Задача 1. Напишите запрос

### Задача 2. Аэропорты с большим числом вылетов

Сравнивать аэропорты по задержкам честно, только если вылетов было достаточно.
Для каждого аэропорта вылета посчитайте по **уже вылетевшим** рейсам:

- **cnt_departed** — число вылетов;
- **avg_delay** — среднюю задержку в минутах, округлённую до одного знака.

Оставьте аэропорты, где вылетов **не меньше 300**. Выведите **departure_airport**, **cnt_departed**,
**avg_delay**, отсортируйте по средней задержке от большей к меньшей.

In [ ]:
%%sql
-- Задача 2. Напишите запрос

### Задача 3. Рейсы по статусам за сентябрь и октябрь

Нужно выяснить, сколько рейсов в каждом статусе было в сентябре и в октябре 2016 года
(месяц — по вылету по расписанию).

Выведите **status**, **cnt_sep** — число рейсов в сентябре, **cnt_oct** — число рейсов в октябре.
Отсортируйте по статусу.

In [ ]:
%%sql
-- Задача 3. Напишите запрос

### Задача 4. Задержки и отмены по неделям

Нужно выяснить, как менялись задержка вылета и количество отменённых рейсов
за всё время в данных — по неделям (неделя — по вылету по расписанию).

Выведите:

- **week_start** — понедельник, с которого начинается неделя, в формате даты (например, 2016-09-12);
- **min_delay**, **avg_delay**, **max_delay** — минимальную, среднюю и максимальную задержку
  вылета в минутах; среднюю округлите до одного знака;
- **cnt_cancelled** — количество отменённых рейсов (статус `Cancelled`).

Отсортируйте по неделям в хронологическом порядке.

In [ ]:
%%sql
-- Задача 4. Напишите запрос

### Задача 5. Задержка рейса и максимальная задержка

Для каждого **вылетевшего** рейса выведите его задержку и рядом — максимальную задержку
по всем рейсам за всё время.

Выведите:

- **flight_id**, **flight_no**;
- **delay** — задержку вылета в минутах;
- **max_delay** — максимальную задержку вылета по всем рейсам в минутах;
- **delay_share_prc** — процент задержки от максимальной, округлённый до целого числа.

Отсортируйте по задержке от большей к меньшей, затем по **flight_id** по возрастанию.
Выведите первые 10 строк.

In [ ]:
%%sql
-- Задача 5. Напишите запрос

### Задача 6. Рейс с наибольшим отклонением от своего аэропорта

Среди рейсов, которые вылетели **12 октября 2016 года** (дата — по вылету по расписанию),
найдите один рейс с наибольшей разницей между его задержкой и средней задержкой вылета
из того же аэропорта. Среднюю по аэропорту считайте по всем его вылетевшим рейсам за всё время.

Выведите:

- **flight_id**, **departure_airport**;
- **delay** — задержку вылета рейса в минутах;
- **avg_airport_delay** — среднюю задержку вылета из того же аэропорта, округлённую до одного знака;
- **diff** — задержку рейса минус среднюю по аэропорту, округлённую до одного знака.

In [ ]:
%%sql
-- Задача 6. Напишите запрос